In [8]:
## import requirement
import numpy as np
import scipy
import csv
import matplotlib.pyplot as plt
!python3 -m pip install py3gpp
from py3gpp import *

## load rx_srsRAN
input_file = '/home/chatchamon/workarea/IQ_constellation/iq_python/input/fllay_092024/waveform_IQComplex_fllay.csv'
waveform = []
with open(input_file, 'r') as f:
    reader = csv.reader(f)
    for row in reader:
      IQComplex = row[0].strip()
      IQComplex = IQComplex.replace('i','j')
      IQComplex = complex(IQComplex)
      waveform.append(IQComplex)

## initial parameter
sampleRate = 15.36e6
nrbSSB = 20
mu = 1
scs = 15 * 2**mu
carrier = nrCarrierConfig(NSizeGrid = nrbSSB, SubcarrierSpacing = scs)
rxOfdmInfo = nrOFDMInfo(carrier)
Nfft = rxOfdmInfo['Nfft']  

# find CFO (coarseFrequencyOffset)
kPSS = np.arange((119-63), (119+64))    # np.arange(56, 183) # check on 3GPP standard 

# With downsampling
syncNfft = 256                  # minimum FFT Size to cover SS burst
syncSR = syncNfft * scs * 1e3

In [9]:
## parameters for loop
# fshifts = [-15k 0 15k]
fshifts = [-15e3, 0, 15e3]
t = np.arange(len(waveform))/sampleRate

## first pair of correlate
# waveform from SDR
coarseFrequencyOffset = fshifts[1]
rxWaveformFreqCorrected = waveform * np.exp(-1j*2*np.pi*coarseFrequencyOffset*t)
rxWaveformDS = scipy.signal.resample_poly(rxWaveformFreqCorrected, syncSR, sampleRate)

